# Cloud plumbing smoke-test

Minimal submission that exercises the **exact crunch `train` / `infer` interface** with a trivial,
fast model and **no heavy dependencies** (no onnx / xgboost / catboost / lightgbm) and
`INFER_PARALLELISM = 1`.

Purpose: isolate whether the **cloud platform** runs at all.
- If this succeeds on the cloud -> the platform is fine; the champion's failure is our heavy stack
  (ONNX RF workers / parallelism / RAM), not the platform.
- If this also fails -> the problem is platform-side (data feed / runner), not our code.

`infer` yields a cheap, non-degenerate score (|online running-mean - history-mean| / history-sd)
so predictions vary and the metric is well-defined, but each point is O(1).

In [ ]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time tLh4vedOjPaByxRSyDPAhufa

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np

In [ ]:
import crunch

crunch_tools = crunch.load_notebook()

In [ ]:
# Local only (stripped from the cloud runner).
train_data, test_data = crunch_tools.load_data()
print(f"train series: {len(train_data)}  |  test series: {len(test_data)}")

## `train()`

Trivial: no learning. Persists a tiny artifact so the train->save->load path is exercised.

In [ ]:
def train(
    datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
    model_directory_path: str,
):
    n = 0
    for _dataset_id, _x_hist, _x_online, _tau in datasets:
        n += 1
    joblib.dump({"ok": True, "n_train_series": n},
                os.path.join(model_directory_path, "model.joblib"))
    print(f"trivial train done ({n} series seen)")

## `infer()`

Generator: first bare `yield` signals readiness, then one cheap score per online point.

In [ ]:
def infer(
    datasets: Iterable[Tuple[List[float], Iterable[float]]],
    model_directory_path: str,
):
    _saved = joblib.load(os.path.join(model_directory_path, "model.joblib"))

    yield  # Signal readiness to the runner.

    for x_historical, x_online in datasets:
        x_h = np.asarray(x_historical, dtype=np.float64)
        hmu = float(x_h.mean()) if len(x_h) else 0.0
        hsd = float(x_h.std()) + 1e-9
        running_sum = 0.0
        count = 0
        for point in x_online:
            running_sum += float(point)
            count += 1
            online_mean = running_sum / count
            yield float(abs(online_mean - hmu) / hsd)

In [ ]:
# @crunch/keep:on
# Single worker: the simplest possible inference path (no fork/parallelism).
INFER_PARALLELISM = 1
print(f"INFER_PARALLELISM={INFER_PARALLELISM}")

## Local test

Runs the same flow the platform will run. Local only.

In [ ]:
crunch_tools.test()

## Submit

On Colab, run the cell below. Locally, download this notebook and upload it to the platform, then create a run.

In [ ]:
Message = "cloud plumbing smoke-test: trivial train/infer, no heavy deps, INFER_PARALLELISM=1"
crunch_tools.submit(message=Message)